In [4]:
import transformers
import peft
import trl
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

In [5]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

In [6]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [11]:
messages = [
    {"role": "system", "content": "당신은 친절하고 전문적인 YourDictionary 고객 서비스 상담원입니다. 사용자의 질문에 정중하게 답변하고 서비스를 안내하세요."},
    {"role": "user", "content": "위 내용을 바탕으로 고객에게 YourDictionary의 문장 생성기 기능을 친절하게 안내해줘."},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

안녕하세요, [이름]님! OurDictionary는 다양한 언어를 포함한 글자와 단어들을 모음으로써 다양한 문장을 만들어주는 기능을 제공합니다. 이를 통해 당신의 텍스트 작성 시 더욱 유창하고 자연스러운 표현을 가능하게 해 드립니다. 또한, 다양한 스타일과 색상 설정을 지원하여 창의적인 콘텐츠를 만들 수 있습니다.

작성 도우미로서 어떤 문장이나 주제가 필요하신가요? 혹은, 당신이 원하는 특정 언어로 문장을 생성하고 싶으시다면 알려주시면 도움이 될 것 같습니다. 함께 문장 생성기를 활용해서 당신의 콘텐츠를 더욱 멋지게 만들어 보세요!<|im_end|>


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 0. CUDA 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 모델과 토크나이저 로드 (device_map 또는 .to(device) 사용)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16, # 메모리 효율을 위해 float16 권장 (GPU 전용)
    device_map=device         # 자동으로 사용 가능한 GPU에 할당
)

test_questions = [
    "YourDictionary의 문장 생성기 기능에 대해 설명해 주세요.",
    "단어의 의미를 더 잘 이해하려면 어떻게 해야 하나요?",
    "문장 생성기 검색창은 어떻게 사용하나요?"
]

base_responses = {}

def generate_response(question):
    messages = [
        {"role": "system", "content": "당신은 친절하고 전문적인 YourDictionary 고객 서비스 상담원입니다. 사용자의 질문에 정중하게 답변하고 서비스를 안내하세요."},
        {"role": "user", "content": question},
    ]
    
    # 2. 입력을 GPU 장치로 이동 (.to(device))
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # 3. 추론 수행
    with torch.no_grad(): # 추론 시 메모리 절약
        outputs = model.generate(**inputs, max_new_tokens=200)
    
    # 결과 디코딩
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response.strip()

# 4. 조건 1~4 수행
for question in test_questions:
    response = generate_response(question)
    base_responses[question] = response
    
    print(f"질문: {question}")
    print(f"답변: {response}")
    print("-" * 30)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

질문: YourDictionary의 문장 생성기 기능에 대해 설명해 주세요.
답변: YourDictionary의 문장 생성기는 다양한 언어와 주제에 대한 원활한 문장을 생성하는 기능을 제공합니다. 이 기능은 다음과 같은 요소들을 포함합니다:

1. **다양한 언어 지원**: YourDictionary는 다양한 언어를 지원하여 영어, 프랑스어, 독일어 등 세계 각국의 언어로의 문장을 생성할 수 있습니다.

2. **고유한 문장 생성**: 이 기능은 특정 주제나 키워드에 따라 고유한 문장을 생성하며, 사용자가 원하시는 내용을 쉽게 찾을 수 있도록 도와줍니다.

3. **대화형 콘텐츠**: 이 기능은 대화형 콘텐츠를 생성하는데 특화되어 있으며, 사용자와의 대화를 통해 자연스러운 의사소통을 유지하도록 설계되었습니다.

4. **편리한 접근성**: �
------------------------------
질문: 단어의 의미를 더 잘 이해하려면 어떻게 해야 하나요?
답변: 당연히 도와드리겠습니다! 단어의 의미를 더 잘 이해하기 위해, 우선 주어진 단어나 문장을 다시 읽는 것이 좋습니다. 이를 통해 어떤 상황이나 개념을 설명하는지 더 명확하게 파악할 수 있습니다.

그 다음으로는 단어의 어근과 접미사를 분석해보세요. 이들은 단어의 기본적인 의미와 연결되어 있는 개념을 나타냅니다. 예를 들어, "un-"는 "비"로 시작하여 "없음", "-able"는 "능하다"로 시작하여 "가능하다"라는 의미를 가집니다.

또한, 이 단어가 사용되는 문맥을 살펴보는 것도 중요합니다. 그 문맥에서 단어가 어떻게 사용되었는지 알아보면 그 단어의 본래의 의미와 연결되게 됩니다.

마지막으로, 다양한 언어 학습 자료나 사
------------------------------
질문: 문장 생성기 검색창은 어떻게 사용하나요?
답변: 문장을 생성하기 위한 검색창을 사용하는 방법은 다음과 같습니다:

1. 웹 브라우저에서 www.youdictionary.com 로 들어갑니다.

2. 페이지의 하단에 위

In [16]:
test_questions = [
    "배송이 언제 오나요?",
    "환불은 어떻게 하나요?",
    "제품이 불량이에요. 어떻게 해야 하나요?",
    "회원 탈퇴는 어떻게 하나요?",
    "적립금은 어떻게 사용하나요?"
]
for question in test_questions:
    # 1: generate_response() 함수를 호출하여 응답 생성
    response = generate_response(question)
    
    # 2: base_responses 딕셔너리에 {질문: 응답} 형태로 저장
    base_responses[question] = response
    
    # 3: 루프 내에서 print() 출력
    # 4: 출력 형식 지정 (질문: {question} / 답변: {response})
    print(f"질문: {question}")
    print(f"답변: {response}")
    print("-" * 50) 

질문: 배송이 언제 오나요?
답변: 죄송합니다, 당장의 배송일자를 알려드리기는 어렵습니다. 현재는 배송예정일을 확인하기 위해 주소를 입력해 주시면 도움을 드리겠습니다. 또한, 주문하신 상품의 상태와 관련된 추가 정보가 필요할 경우 말씀해 주세요.
--------------------------------------------------
질문: 환불은 어떻게 하나요?
답변: 안녕하세요! 환불 관련事宜에 대해 알려드릴 수 있습니다.

1. **환불 요청**: 환불을 원하는 경우, 해당 제품 또는 서비스가 제공된 날로부터 30일 이내에 고객센터로 연락하여 환불을 신청해야 합니다. 사전 계약이 있는 경우에는 계약 체결 시 주의사항을 확인하시고 계약서에 명시된 조건을 준수해야 합니다.

2. **확인 절차**: 환불 신청 시, 회사에서 제공하는 문서나 정보를 확인하고 필요한 서류를 준비해야 합니다. 예상되는 시간대와 방법 등에 대한 추가 정보도 필요할 수 있습니다.

3. **반려 사유 확인**: 환불이 승인되면, 반려 사유를 확인받아야 합니다. 이는 고객에게 불편함을 줄여주기 위해 필요할 수도 있습니다.
--------------------------------------------------
질문: 제품이 불량이에요. 어떻게 해야 하나요?
답변: 안녕하세요, 저희가 도와드릴 수 있는 방법을 알려주시겠어요? 제품의 문제나 상태에 대해 좀 더 자세히 설명해 주시면 감사하겠습니다.
--------------------------------------------------
질문: 회원 탈퇴는 어떻게 하나요?
답변: 죄송합니다, 하지만 이 기술 지원 시스템에서는 회원 탈퇴나 관련된 정보를 제공할 수 없습니다. 회원이탈을 원하시는 경우에는 해당 사이트 또는 앱의 관리자에게 연락하시거나, 그들의 문의처로 연결해 주시면 됩니다.
--------------------------------------------------
질문: 적립금은 어떻게 사용하나요?
답변: 저희 YourDic

In [17]:
# 고객 서비스 학습 데이터
# 실제로는 더 많은 데이터가 필요합니다 (최소 50개 이상 권장)
training_data = [
    # 배송 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "배송이 언제 오나요?",
        "output": "안녕하세요, 고객님! 주문하신 상품은 결제 완료 후 영업일 기준 2~3일 내에 배송됩니다. 정확한 배송 현황은 마이페이지 > 주문내역에서 운송장 번호로 조회하실 수 있습니다. 추가 문의사항이 있으시면 말씀해 주세요!"
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "배송 조회는 어디서 하나요?",
        "output": "배송 조회는 마이페이지 > 주문내역에서 가능합니다. 해당 주문의 '배송조회' 버튼을 클릭하시면 택배사 배송 추적 페이지로 이동합니다. 운송장 번호가 아직 등록되지 않았다면 발송 준비 중이니 조금만 기다려 주세요!"
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "해외 배송도 되나요?",
        "output": "네, 해외 배송 가능합니다! 해외 배송은 국가별로 배송비와 소요 기간이 다르며, 주문 시 배송지 국가를 선택하시면 예상 배송비를 확인하실 수 있습니다. 보통 아시아 지역은 7~10일, 미주/유럽은 2~3주 정도 소요됩니다."
    },
    # 환불 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "환불은 어떻게 하나요?",
        "output": "환불 절차를 안내해 드리겠습니다. 마이페이지 > 주문내역에서 해당 주문의 '환불 신청' 버튼을 클릭해 주세요. 상품 수령 후 7일 이내, 미개봉 상태라면 100% 환불이 가능합니다. 환불 처리는 신청 후 영업일 기준 3~5일 내에 완료됩니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "환불 처리 기간은 얼마나 걸리나요?",
        "output": "환불 신청이 승인되면 결제 수단에 따라 처리 기간이 다릅니다. 신용카드는 영업일 기준 3~5일, 계좌이체는 1~2일, 포인트 결제는 즉시 복원됩니다. 환불 진행 상황은 마이페이지에서 확인하실 수 있습니다."
    },
    # 교환/불량 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "제품이 불량이에요. 어떻게 해야 하나요?",
        "output": "불편을 드려 정말 죄송합니다. 불량 제품은 무료로 교환 또는 환불해 드립니다. 불량 부분이 보이도록 사진을 찍어 고객센터(1234-5678)로 보내주시면, 확인 후 즉시 새 제품을 발송해 드리겠습니다. 반품 택배비도 저희가 부담합니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "교환 신청은 어떻게 하나요?",
        "output": "교환 신청은 마이페이지 > 주문내역에서 가능합니다. 교환하실 상품을 선택하고 '교환 신청' 버튼을 클릭한 뒤, 교환 사유와 원하시는 옵션(사이즈, 색상 등)을 선택해 주세요. 단순 변심 교환 시 왕복 배송비 5,000원이 발생합니다."
    },
    # 회원 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "회원 탈퇴는 어떻게 하나요?",
        "output": "회원 탈퇴는 마이페이지 > 회원정보 > 회원탈퇴에서 진행하실 수 있습니다. 탈퇴 시 보유하신 적립금과 쿠폰은 모두 소멸되며, 주문 내역은 관련 법률에 따라 일정 기간 보관 후 삭제됩니다. 탈퇴 전 확인해 주세요!"
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "비밀번호를 잊어버렸어요.",
        "output": "비밀번호 분실 시 로그인 페이지의 '비밀번호 찾기'를 클릭해 주세요. 가입하신 이메일로 비밀번호 재설정 링크가 발송됩니다. 메일이 오지 않으면 스팸함을 확인하시거나, 고객센터로 연락 주시면 도와드리겠습니다."
    },
    # 적립금/쿠폰 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "적립금은 어떻게 사용하나요?",
        "output": "적립금은 주문 결제 시 현금처럼 사용하실 수 있습니다. 결제 페이지에서 '적립금 사용' 항목에 사용하실 금액을 입력해 주세요. 단, 최소 주문금액 10,000원 이상일 때 사용 가능하며, 적립금은 최대 결제금액의 10%까지 사용 가능합니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "쿠폰 등록은 어디서 하나요?",
        "output": "쿠폰 등록은 마이페이지 > 쿠폰함에서 하실 수 있습니다. '쿠폰 등록' 버튼을 클릭하고 쿠폰 코드를 입력해 주세요. 등록된 쿠폰은 결제 시 자동으로 적용 가능한 목록에 표시됩니다. 쿠폰마다 사용 조건이 다르니 유효기간도 확인해 주세요!"
    },
    # 주문 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "주문 취소는 어떻게 하나요?",
        "output": "주문 취소는 '배송 준비 중' 상태 전까지 가능합니다. 마이페이지 > 주문내역에서 해당 주문의 '주문 취소' 버튼을 클릭해 주세요. 이미 발송된 경우에는 수령 후 반품 절차를 진행해 주시면 됩니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "주문 내역은 어디서 확인하나요?",
        "output": "주문 내역은 마이페이지 > 주문내역에서 확인하실 수 있습니다. 주문 날짜, 상품 정보, 결제 금액, 배송 상태를 한눈에 보실 수 있습니다. 비회원 주문의 경우 '비회원 주문조회'에서 주문번호와 연락처로 조회 가능합니다."
    },
    # 결제 관련
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "어떤 결제 수단을 사용할 수 있나요?",
        "output": "다양한 결제 수단을 지원합니다! 신용카드, 체크카드, 계좌이체, 무통장입금, 카카오페이, 네이버페이, 토스 등 간편결제도 모두 사용 가능합니다. 무통장입금은 주문 후 24시간 내에 입금해 주셔야 자동 취소되지 않습니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "카드 할부 가능한가요?",
        "output": "네, 신용카드 할부 결제 가능합니다! 5만원 이상 결제 시 2~12개월 할부를 선택하실 수 있습니다. 무이자 할부 행사는 카드사별로 다르니, 결제 페이지에서 해당 카드사의 무이자 혜택을 확인해 주세요."
    },
    # 기타
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "고객센터 운영 시간이 어떻게 되나요?",
        "output": "고객센터는 평일 오전 9시부터 오후 6시까지 운영됩니다. 점심시간(12시~1시)에는 상담이 어려울 수 있습니다. 주말 및 공휴일은 휴무이며, 1:1 문의 게시판은 24시간 접수 가능합니다. 전화번호는 1234-5678입니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "영수증 발급 가능한가요?",
        "output": "네, 영수증 발급 가능합니다! 마이페이지 > 주문내역에서 해당 주문의 '영수증 출력' 버튼을 클릭하시면 됩니다. 현금영수증은 결제 시 등록하신 번호로 자동 발급되며, 세금계산서가 필요하시면 고객센터로 사업자등록증을 보내주세요."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "재입고 알림 받을 수 있나요?",
        "output": "네, 품절 상품의 재입고 알림을 받으실 수 있습니다! 해당 상품 페이지에서 '재입고 알림 신청' 버튼을 클릭하고 연락처를 등록해 주세요. 재입고 시 SMS 또는 카카오톡으로 알림을 보내드립니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "포장 선물 가능한가요?",
        "output": "네, 선물 포장 서비스를 제공하고 있습니다! 주문 시 '선물 포장' 옵션을 선택해 주시면 예쁜 선물 박스와 리본으로 포장해 드립니다. 포장비는 2,000원이며, 메시지 카드도 무료로 동봉 가능합니다."
    },
    # 추가 데이터
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "배송비는 얼마인가요?",
        "output": "기본 배송비는 3,000원이며, 3만원 이상 구매 시 무료 배송입니다! 도서산간 지역은 추가 배송비 3,000원이 발생할 수 있습니다. 무거운 대형 상품은 별도 배송비가 적용되며, 상품 페이지에서 확인하실 수 있습니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "상품 문의는 어디서 하나요?",
        "output": "상품 문의는 해당 상품 페이지 하단의 'Q&A' 게시판에서 하실 수 있습니다. 상품 상세 정보, 사이즈, 재질 등 궁금한 점을 남겨주시면 담당자가 빠르게 답변드립니다. 개인정보가 포함된 문의는 1:1 문의를 이용해 주세요."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "주문 수정이 가능한가요?",
        "output": "주문 수정은 '결제 완료' 상태일 때만 가능합니다. 마이페이지 > 주문내역에서 배송지 변경이나 옵션 변경을 하실 수 있습니다. 이미 '배송 준비 중'으로 변경되었다면 주문 취소 후 재주문 부탁드립니다."
    },
    {
        "instruction": "당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.",
        "input": "앱에서도 주문할 수 있나요?",
        "output": "네, 모바일 앱에서도 편리하게 주문하실 수 있습니다! 앱스토어 또는 구글플레이에서 저희 앱을 다운로드해 주세요. 앱 전용 할인 쿠폰과 푸시 알림 혜택도 받으실 수 있습니다."
    }
]

In [18]:
def formatting_func(example):
    # 조건 1 & 2: messages 리스트 생성 (system, user, assistant 구성)
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]}
    ]
    
    # 조건 3: apply_chat_template을 사용하여 텍스트 포맷팅 (학습용이므로 tokenize=False)
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    
    # 조건 4: {"text": formatted_text} 형태의 딕셔너리 반환
    return {"text": formatted_text}

# 실제 데이터 적용 예시 (리스트 컴프리헨션 사용)
formatted_dataset = [formatting_func(data) for data in training_data]

# 변환 결과 확인 (첫 번째 데이터)
print(formatted_dataset[0]["text"])

<|im_start|>system
당신은 친절한 고객 서비스 상담원입니다. 고객의 질문에 정확하고 도움이 되는 답변을 제공하세요.<|im_end|>
<|im_start|>user
배송이 언제 오나요?<|im_end|>
<|im_start|>assistant
안녕하세요, 고객님! 주문하신 상품은 결제 완료 후 영업일 기준 2~3일 내에 배송됩니다. 정확한 배송 현황은 마이페이지 > 주문내역에서 운송장 번호로 조회하실 수 있습니다. 추가 문의사항이 있으시면 말씀해 주세요!<|im_end|>



In [19]:
# 1~3번 조건: count_tokens() 함수 정의
def count_tokens(example):
    # 조건 1: tokenizer(example["text"])를 사용하여 토큰화
    tokenized = tokenizer(example["text"])
    
    # 조건 2: input_ids 길이를 len()으로 계산
    token_length = len(tokenized["input_ids"])
    
    # 조건 3: {"token_count": 토큰_길이} 형태의 딕셔너리 반환
    return {"token_count": token_length}

# 위에서 만든 formatted_dataset에 토큰 카운트 추가
# (리스트 형태일 경우를 가정한 분석 예시입니다)
token_counts = [count_tokens(data)["token_count"] for data in formatted_dataset]

# 조건 4: 분석 결과(평균, 최대, 최소) 출력
if token_counts:
    avg_len = sum(token_counts) / len(token_counts)
    max_len = max(token_counts)
    min_len = min(token_counts)

    print(f"평균 토큰 길이: {avg_len:.2f}")
    print(f"최대 토큰 길이: {max_len}")
    print(f"최소 토큰 길이: {min_len}")

평균 토큰 길이: 135.78
최대 토큰 길이: 154
최소 토큰 길이: 119


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 조건 1, 2, 3: 모델 로드 설정
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # 조건 2: 16비트 정밀도
    device_map="auto",         # 조건 3: 자동 디바이스 매핑
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 조건 4: 모델 로드 후 메모리 사용량 출력
def print_memory_usage():
    if torch.cuda.is_available():
        # 현재 할당된 메모리와 예약된 메모리 계산 (GB 단위)
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        
        print(f"--- GPU 메모리 사용량 ---")
        print(f"현재 할당된 메모리: {allocated:.2f} GB")
        print(f"현재 예약된 메모리: {reserved:.2f} GB")
    else:
        print("CUDA를 사용할 수 없는 환경입니다.")

print_memory_usage()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

--- GPU 메모리 사용량 ---
현재 할당된 메모리: 1.40 GB
현재 예약된 메모리: 4.31 GB


In [21]:
from peft import LoraConfig, get_peft_model, TaskType

# 1~3번 조건: LoraConfig 설정 객체 생성
lora_config = LoraConfig(
    r=16,                                    # 조건 2: LoRA 랭크 설정
    lora_alpha=32,                           # 조건 2: 스케일링 계수
    lora_dropout=0.05,                       # 조건 2: 드롭아웃 확률
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # 조건 3: 대상 레이어 지정
    task_type=TaskType.CAUSAL_LM,           # 조건 4: 인과적 언어 모델 태스크
    bias="none"
)

# 생성된 설정을 모델에 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인 (선택 사항)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [32]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# 1. list 형태의 데이터를 Hugging Face Dataset 객체로 변환
# formatted_dataset이 [{"text": "..."}, {"text": "..."}] 구조여야 합니다.
hf_train_dataset = Dataset.from_list(formatted_dataset)

# 2. SFTConfig 설정 (조건 1~3 포함)
sft_config = SFTConfig(
    output_dir="./outputs",
    num_train_epochs=3,               # 조건 1
    per_device_train_batch_size=2,    # 조건 2
    gradient_accumulation_steps=4,    # 조건 3
    dataset_text_field="text",        
    #max_seq_length=512,               
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# 3. 조건 4: 실효 배치 크기 출력
effective_batch_size = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
print(f"실효 배치 크기(Effective Batch Size): {effective_batch_size}")

# 4. SFTTrainer 초기화 (변환된 hf_train_dataset 사용)
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train_dataset, # list 대신 Dataset 객체 전달
    args=sft_config,
)

# 학습 시작
# trainer.train()

실효 배치 크기(Effective Batch Size): 8


Adding EOS to train dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

In [41]:
train_result = trainer.train()
# log_history에서 'loss'가 포함된 항목만 필터링
epoch_logs = [log for log in trainer.state.log_history if 'loss' in log]

if len(epoch_logs) >= 2:
    # 초기 에폭(첫 번째 로그)과 최종 에폭(마지막 로그) 정보 가져오기
    initial_epoch = epoch_logs[0]
    final_epoch = epoch_logs[-1]
    
    initial_loss = initial_epoch['loss']
    final_loss = final_epoch['loss']
    
    # Loss 감소량 및 감소율 계산
    loss_diff = initial_loss - final_loss
    reduction_rate = (loss_diff / initial_loss) * 100

    # 결과 출력 (주피터 셀 출력용)
    print("\n" + "="*40)
    print("   [학습 전/후 Loss 비교 분석]")
    print("="*40)
    print(f"▶ 초기 에폭 (Epoch {round(initial_epoch['epoch'])}): Loss = {initial_loss:.4f}")
    print(f"▶ 최종 에폭 (Epoch {round(final_epoch['epoch'])}): Loss = {final_loss:.4f}")
    print("-" * 40)
    print(f"▶ 전체 감소량: {loss_diff:.4f}")
    print(f"▶ Loss 감소율: {reduction_rate:.2f}%")
    print("="*40)
else:
    print("로그 데이터가 부족하여 비교 분석을 수행할 수 없습니다.")

# 조건 3: 최종 메트릭 출력
metrics = train_result.metrics
print(f"\n[최종 통계] 시간: {metrics['train_runtime']:.2f}s | 전체 평균 Loss: {metrics['train_loss']:.4f}")

Step,Training Loss


로그 데이터가 부족하여 비교 분석을 수행할 수 없습니다.

[최종 통계] 시간: 11.83s | 전체 평균 Loss: 0.2284


In [42]:
# 1. 조건 1~3: Fine-tuned 모델용 응답 생성 함수 정의
def generate_response_finetuned(question):
    # 조건 2: Q1과 동일한 Chat Template 방식 사용
    messages = [
        {"role": "system", "content": "당신은 친절하고 전문적인 YourDictionary 고객 서비스 상담원입니다. 사용자의 질문에 정중하게 답변하고 서비스를 안내하세요."},
        {"role": "user", "content": question},
    ]
    
    # 템플릿 적용 및 토큰화
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # 조건 3: model.generate() 호출 및 결과 디코딩
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200)
    
    # 응답 텍스트 추출 (입력 토큰 제외)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response.strip()

# 파이튜닝 후 응답 저장용 딕셔너리
finetuned_responses = {}

# 2. 조건 4: 모든 질문에 대해 전/후 응답 생성 및 비교 출력
print("\n" + "="*80)
print("   [Fine-tuning 전/후 성능 비교 분석]")
print("="*80)

for question in test_questions:
    # 파이튜닝 후 응답 생성
    ft_response = generate_response_finetuned(question)
    finetuned_responses[question] = ft_response
    
    # 이전 답변(base_responses)과 나란히 출력하여 비교
    print(f"▶ 질문: {question}")
    print(f"--- [Fine-tuning 전 (Base)] ---")
    print(f"{base_responses.get(question, '이전 기록 없음')}")
    print(f"--- [Fine-tuning 후 (LoRA)] ---")
    print(f"{ft_response}")
    print("-" * 80)

print("="*80)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.



   [Fine-tuning 전/후 성능 비교 분석]
▶ 질문: 배송이 언제 오나요?
--- [Fine-tuning 전 (Base)] ---
죄송합니다, 당장의 배송일자를 알려드리기는 어렵습니다. 현재는 배송예정일을 확인하기 위해 주소를 입력해 주시면 도움을 드리겠습니다. 또한, 주문하신 상품의 상태와 관련된 추가 정보가 필요할 경우 말씀해 주세요.
--- [Fine-tuning 후 (LoRA)] ---
안은 20

# coding systems and the system.
- 199999999月球系系统系统的正确答案。
A group is a single system of the value of the average, which
Cyls = 20000
Determine the number of the
B
C. The given by the\n\n\n\n\n\\section . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
--------------------------------------------------------------------------------
▶ 질문: 환불은 어떻게 하나요?
--- [Fine-tuning 전 (Base)] ---
안녕하세요! 환불 관련事宜에 대해 알려드릴 수 있습니다.

1. **환불 요청**: 환불을 원하는 경우, 해당 제품 또는 서비스가 제공된 날로부터 30일 이내에 고객센터로 연락하여 환불을 신청해야 합니다. 사전 계약이 있는 경우에는 계약 체결 시 주의사항을 확인하시고 계약서에 명시된 조건을 준수해야 합니다.

2. **확인 절차**: 환불 신청

In [43]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_PATH = "./lora_adapter_checkpoint"

# --- [조건 1 & 2] LoRA 어댑터 및 토크나이저 저장 ---
# 학습된 trainer 객체로부터 모델(어댑터) 저장
trainer.save_model(ADAPTER_PATH)
# 토크나이저 설정 저장
tokenizer.save_pretrained(ADAPTER_PATH)

# --- [조건 4] 저장된 파일 목록 및 용량 출력 ---
print(f"\n[저장된 파일 목록 및 용량 확인: {ADAPTER_PATH}]")
print("-" * 50)
if os.path.exists(ADAPTER_PATH):
    for filename in os.listdir(ADAPTER_PATH):
        file_path = os.path.join(ADAPTER_PATH, filename)
        # 용량을 MB 단위로 계산
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"- {filename:30} | {file_size_mb:.2f} MB")
print("-" * 50)

# --- [조건 3] 모델 로드 함수 정의 ---
def load_finetuned_model(base_model_id, adapter_path):
    # 1. 베이스 모델 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # 2. 저장된 LoRA 어댑터 적용
    # PeftModel.from_pretrained를 사용하여 베이스 모델 위에 어댑터를 올립니다.
    ft_model = PeftModel.from_pretrained(base_model, adapter_path)
    
    # 3. 토크나이저 로드
    ft_tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    
    return ft_model, ft_tokenizer

# 함수 호출 예시 (모델 ID는 기존 사용하던 "Qwen/Qwen2.5-1.5B-Instruct" 사용)
# loaded_model, loaded_tokenizer = load_finetuned_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_PATH)


[저장된 파일 목록 및 용량 확인: ./lora_adapter_checkpoint]
--------------------------------------------------
- adapter_config.json            | 0.00 MB
- training_args.bin              | 0.00 MB
- chat_template.jinja            | 0.00 MB
- tokenizer.json                 | 10.89 MB
- README.md                      | 0.00 MB
- adapter_model.safetensors      | 16.65 MB
- tokenizer_config.json          | 0.00 MB
--------------------------------------------------
